# Python データ分析テンプレート

データ分析でよく使うコードをセクション別にまとめたテンプレート。  
上から順に実行しながら進める。各セルは独立して動くように設計。

---
**目次**
- [0. ライブラリのインポート](#0)
- [1. データ読み込み](#1)
- [2. データの概要確認](#2)
- [3. データクレンジング](#3)
- [4. データ変換・加工](#4)
- [5. 集計・グルーピング](#5)
- [6. 可視化](#6)
- [7. 統計分析](#7)
- [8. 機械学習](#8)
- [9. 結果の保存](#9)

<a id='0'></a>
## 0. ライブラリのインポート

分析で必須のライブラリを一括インポートする。  
日本語フォントと表示オプションもここで設定しておく。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings

# 日本語フォント
matplotlib.rcParams['font.family'] = 'MS Gothic'   # Windows
# matplotlib.rcParams['font.family'] = 'Hiragino Sans'  # Mac

# DataFrame の表示設定
pd.set_option('display.max_columns', 50)           # 最大列数
pd.set_option('display.max_rows', 100)             # 最大行数
pd.set_option('display.float_format', '{:.2f}'.format)  # 小数2桁
warnings.filterwarnings('ignore')
%matplotlib inline

print('OK')

<a id='1'></a>
## 1. データ読み込み

ファイル形式によって読み込み方が異なる。  
実際のデータに合わせてセルを選んで実行する。

### CSV

In [ ]:
df = pd.read_csv('data.csv')
df.head()

In [ ]:
# 文字化けしたとき / オプション指定
df = pd.read_csv(
    'data.csv',
    encoding='shift_jis',   # 文字コード (utf-8 / shift_jis / cp932)
    sep='\t',               # 区切り文字 (デフォルトはカンマ)
    skiprows=2,             # 先頭2行をスキップ
    nrows=1000,             # 先頭1000行だけ読む
    parse_dates=['date'],   # 日付列を自動でdatetime型に変換
    index_col=0,            # 0列目をインデックスにする
)
df.head()

### Excel

In [ ]:
df = pd.read_excel('data.xlsx', sheet_name='Sheet1')
df.head()

In [ ]:
# 全シートを dict 形式で読む
sheets = pd.read_excel('data.xlsx', sheet_name=None)
print(list(sheets.keys()))   # シート名の一覧
df = sheets['Sheet1']

### JSON / SQLite

In [ ]:
# JSON
df = pd.read_json('data.json', orient='records')
df.head()

In [ ]:
# SQLite
import sqlite3
conn = sqlite3.connect('database.db')
df = pd.read_sql('SELECT * FROM table_name LIMIT 1000', conn)
conn.close()
df.head()

### サンプルデータ（動作確認用）

実データがない場合は sklearn のサンプルデータを使う。  
以降のセルはこのデータで動作確認できる。

In [ ]:
from sklearn.datasets import load_iris
iris = load_iris(as_frame=True)
df = iris.frame
df.columns = ['がく片長', 'がく片幅', '花弁長', '花弁幅', '品種']

print(f'行数: {len(df):,}  列数: {df.shape[1]}')
df.head()

<a id='2'></a>
## 2. データの概要確認

読み込んだ直後に必ず確認する項目。  
形状・型・欠損・重複を把握してからクレンジングに進む。

### 形状・型・基本情報

In [ ]:
print('shape:', df.shape)
print()
df.info()

### 統計量

In [ ]:
# 数値列の統計量
df.describe()

In [ ]:
# カテゴリ列も含めた統計量
df.describe(include='all')

### 欠損値

In [ ]:
# 欠損数と欠損率を一覧表示（欠損がある列だけ表示）
missing = pd.DataFrame({
    '欠損数':   df.isnull().sum(),
    '欠損率(%)': (df.isnull().mean() * 100).round(1)
})
display(missing[missing['欠損数'] > 0])
print(f'欠損なし列数: {(missing["欠損数"] == 0).sum()} / {len(missing)}')

### 重複行

In [ ]:
print(f'重複行数: {df.duplicated().sum()}')
df[df.duplicated(keep=False)].head()

### カテゴリ列のユニーク値

In [ ]:
cat_cols = df.select_dtypes(include=['object', 'category']).columns
for col in cat_cols:
    print(f'\n--- {col}  ({df[col].nunique()} 種類) ---')
    print(df[col].value_counts())

<a id='3'></a>
## 3. データクレンジング

欠損・重複・型のズレを修正して分析できる状態に整える。

### 欠損値の処理

In [ ]:
# 欠損行を削除（全列に欠損がないことを条件）
df = df.dropna()
print(df.shape)

In [ ]:
# 特定列に欠損がある行だけ削除
df = df.dropna(subset=['がく片長', '花弁長'])
print(df.shape)

In [ ]:
# 数値列: 平均・中央値・0 で補完
df['がく片長'] = df['がく片長'].fillna(df['がく片長'].mean())    # 平均
# df['がく片長'] = df['がく片長'].fillna(df['がく片長'].median()) # 中央値
# df['がく片長'] = df['がく片長'].fillna(0)                       # 0固定

In [ ]:
# カテゴリ列: 最頻値で補完
df['品種'] = df['品種'].fillna(df['品種'].mode()[0])

In [ ]:
# 時系列データ: 前後の値で補完
df['がく片長'] = df['がく片長'].ffill()  # 前の値で埋める (前方補完)
# df['がく片長'] = df['がく片長'].bfill()  # 後の値で埋める (後方補完)

### 重複行の削除

In [ ]:
before = len(df)
df = df.drop_duplicates()                                  # 完全一致の重複を削除
# df = df.drop_duplicates(subset=['がく片長'], keep='last') # 特定列で判定・後ろを残す
print(f'{before - len(df)} 行削除 → {len(df)} 行')

### 型変換

In [ ]:
# 数値型への変換
# df['列名'] = df['列名'].astype(int)
# df['列名'] = df['列名'].astype(float)
# df['列名'] = pd.to_numeric(df['列名'], errors='coerce')  # 変換できない値はNaN

# 日付型への変換
# df['日付'] = pd.to_datetime(df['日付'], format='%Y/%m/%d')
# df['日付'] = pd.to_datetime(df['日付'], format='%Y%m%d')

# カテゴリ型 (メモリ節約)
# df['品種'] = df['品種'].astype('category')

print(df.dtypes)

### 文字列の整形

In [ ]:
# よく使う文字列クレンジング
# df['列名'] = df['列名'].str.strip()                            # 前後の空白除去
# df['列名'] = df['列名'].str.replace('　', '', regex=False)     # 全角スペース除去
# df['列名'] = df['列名'].str.lower()                            # 小文字化
# df['列名'] = df['列名'].str.upper()                            # 大文字化
# df['列名'] = df['列名'].str.replace(r'\s+', '', regex=True)   # 空白を全除去
# df['列名'] = df['列名'].str.replace(r'[^\d]', '', regex=True) # 数字以外を除去
# df['列名'] = df['列名'].str.zfill(5)                          # ゼロ埋め5桁

print('文字列整形: 必要な行のコメントを外して実行')

<a id='4'></a>
## 4. データ変換・加工

新しい列の追加・条件分岐・日付操作・絞り込みなど。

### 列の追加・計算

In [ ]:
# 既存列の計算で新列を作る
df['がく比'] = df['がく片長'] / df['がく片幅']         # 割り算
df['がく合計'] = df['がく片長'] + df['がく片幅']       # 足し算

# 関数を適用 (apply)
df['花弁長_log'] = df['花弁長'].apply(np.log1p)        # log変換

df[['がく比', 'がく合計', '花弁長_log']].head()

### 条件分岐で新列を作成

In [ ]:
# 2値: np.where
df['大小'] = np.where(df['花弁長'] >= 4, '大', '小')
df['大小'].value_counts()

In [ ]:
# 多値: np.select
conditions = [
    df['花弁長'] >= 6,
    df['花弁長'] >= 4,
    df['花弁長'] >= 2,
]
choices = ['L', 'M', 'S']
df['サイズ'] = np.select(conditions, choices, default='XS')
df['サイズ'].value_counts()

### 日付操作

In [ ]:
# 日付型の列があるときに使う
# df['日付'] = pd.to_datetime(df['日付'])  # まず変換

# df['年']     = df['日付'].dt.year
# df['月']     = df['日付'].dt.month
# df['日']     = df['日付'].dt.day
# df['曜日']   = df['日付'].dt.day_name()      # 'Monday' など英語
# df['曜日番号'] = df['日付'].dt.dayofweek     # 0=月, 6=日
# df['週番号'] = df['日付'].dt.isocalendar().week
# df['四半期'] = df['日付'].dt.quarter
# df['経過日数'] = (pd.Timestamp.today() - df['日付']).dt.days

print('日付列があれば上のコメントを外して実行')

### フィルタリング (行の絞り込み)

In [ ]:
# 単条件
df_large = df[df['花弁長'] >= 5]
print(len(df_large))

In [ ]:
# AND条件 (&) / OR条件 (|)
df_fil = df[(df['花弁長'] >= 4) & (df['花弁幅'] >= 1.5)]
print(len(df_fil))

In [ ]:
# isin: リストに含まれる行
df_fil = df[df['品種'].isin([0, 2])]
print(len(df_fil))

In [ ]:
# query メソッド (可読性が高い)
df_fil = df.query('花弁長 >= 5 and 品種 == 2')
print(len(df_fil))

### ビニング (数値 → カテゴリ)

In [ ]:
# cut: 境界値を自分で指定
df['花弁区分'] = pd.cut(
    df['花弁長'],
    bins=[0, 2, 4, 6, 10],
    labels=['XS', 'S', 'M', 'L']
)
df['花弁区分'].value_counts()

In [ ]:
# qcut: 等頻度で自動分割
df['花弁クラス'] = pd.qcut(df['花弁長'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
df['花弁クラス'].value_counts()

<a id='5'></a>
## 5. 集計・グルーピング

`groupby` でグループごとに集計する。  
複数の集計を同時に行うときは `agg` を使う。

### groupby の基本

In [ ]:
# グループ別の平均
df.groupby('品種').mean(numeric_only=True).round(2)

In [ ]:
# 複数の集計関数を同時に実行 (agg)
df.groupby('品種').agg(
    件数      = ('花弁長', 'count'),
    平均花弁長 = ('花弁長', 'mean'),
    最大花弁長 = ('花弁長', 'max'),
    最小花弁長 = ('花弁長', 'min'),
    標準偏差   = ('花弁長', 'std'),
).round(2)

In [ ]:
# 複数キーでグルーピング
df.groupby(['品種', 'サイズ'])['花弁長'].mean().round(2).unstack()

### ピボットテーブル

In [ ]:
pivot = pd.pivot_table(
    df,
    values='花弁長',         # 集計する列
    index='品種',            # 行ラベル
    columns='花弁クラス',    # 列ラベル
    aggfunc='mean',          # 集計関数 (mean / sum / count)
    fill_value=0             # 空欄を0で埋める
).round(2)
pivot

### テーブル結合 (merge)

In [ ]:
# 結合のイメージ (実際のデータに合わせてキー列を変える)

# left join: df_main を基準に df_sub を結合
# merged = pd.merge(df_main, df_sub, on='キー列', how='left')

# inner join: 両方にある行だけ残す
# merged = pd.merge(df_main, df_sub, on='キー列', how='inner')

# キー列名が違う場合
# merged = pd.merge(df_main, df_sub, left_on='id', right_on='user_id', how='left')

print('結合するDataFrameを準備してから実行')

### 縦結合 (concat)

In [ ]:
# 同じ列構造のDataFrameを縦に積む
# combined = pd.concat([df1, df2, df3], ignore_index=True)

# 動作確認
df_a = df[:50].copy()
df_b = df[50:].copy()
combined = pd.concat([df_a, df_b], ignore_index=True)
print(combined.shape)

<a id='6'></a>
## 6. 可視化

matplotlib / seaborn でグラフを描く。  
seaborn はシンプルに書けて見た目もきれい。

### ヒストグラム

In [ ]:
plt.figure(figsize=(6, 4))
df['花弁長'].hist(bins=20, color='steelblue', edgecolor='white')
plt.title('花弁長の分布')
plt.xlabel('花弁長')
plt.ylabel('件数')
plt.tight_layout()
plt.show()

In [ ]:
# グループ別に重ねる
plt.figure(figsize=(7, 4))
for label, group in df.groupby('品種'):
    group['花弁長'].hist(bins=15, alpha=0.6, label=f'品種{label}')
plt.title('品種別 花弁長の分布')
plt.xlabel('花弁長')
plt.ylabel('件数')
plt.legend()
plt.tight_layout()
plt.show()

### 散布図

In [ ]:
plt.figure(figsize=(6, 5))
sns.scatterplot(data=df, x='花弁長', y='花弁幅', hue='品種', alpha=0.8, palette='Set1')
plt.title('花弁長 vs 花弁幅')
plt.tight_layout()
plt.show()

### 棒グラフ

In [ ]:
# グループ別の平均
agg = df.groupby('品種')[['花弁長', '花弁幅']].mean()
agg.plot(kind='bar', figsize=(7, 4), color=['steelblue', 'coral'], edgecolor='white')
plt.title('品種別 平均値')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# 積み上げ棒グラフ
pivot_cnt = df.groupby(['品種', 'サイズ']).size().unstack(fill_value=0)
pivot_cnt.plot(kind='bar', stacked=True, figsize=(6, 4))
plt.title('品種 × サイズ別 件数')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 箱ひげ図 / バイオリンプロット

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(data=df, x='品種', y='花弁長', palette='Set2', ax=axes[0])
axes[0].set_title('箱ひげ図')

sns.violinplot(data=df, x='品種', y='花弁長', palette='Set2', ax=axes[1])
axes[1].set_title('バイオリンプロット')

plt.tight_layout()
plt.show()

### 相関ヒートマップ

In [ ]:
corr = df.select_dtypes(include='number').corr()

plt.figure(figsize=(6, 5))
sns.heatmap(
    corr,
    annot=True, fmt='.2f',
    cmap='coolwarm', vmin=-1, vmax=1,
    square=True, linewidths=0.5
)
plt.title('相関係数ヒートマップ')
plt.tight_layout()
plt.show()

### ペアプロット

In [ ]:
# 全数値列の組み合わせをまとめて可視化
num_cols = df.select_dtypes(include='number').columns.tolist()
sns.pairplot(df[num_cols + ['品種']], hue='品種', corner=True, plot_kws={'alpha': 0.6})
plt.suptitle('ペアプロット', y=1.02)
plt.show()

### 折れ線グラフ (時系列)

In [ ]:
# 日付列と値列があるときに使う
# plt.figure(figsize=(12, 4))
# plt.plot(df['日付'], df['値'], marker='o', markersize=3, linewidth=1)
# plt.title('時系列グラフ')
# plt.xlabel('日付')
# plt.ylabel('値')
# plt.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.show()

print('時系列: 日付列と値列を指定して実行')

### 円グラフ

In [ ]:
counts = df['品種'].value_counts()
plt.figure(figsize=(5, 5))
plt.pie(
    counts,
    labels=[f'品種{i}' for i in counts.index],
    autopct='%1.1f%%',
    startangle=90,
    colors=sns.color_palette('Set2', len(counts))
)
plt.title('品種別構成比')
plt.show()

<a id='7'></a>
## 7. 統計分析

仮説検定・相関分析など。`scipy.stats` を使う。

### 基本統計量

In [ ]:
from scipy import stats

col = '花弁長'
print(f'平均値   : {df[col].mean():.4f}')
print(f'中央値   : {df[col].median():.4f}')
print(f'標準偏差 : {df[col].std():.4f}')
print(f'分散     : {df[col].var():.4f}')
print(f'歪度     : {df[col].skew():.4f}  (正: 右に裾が長い)')
print(f'尖度     : {df[col].kurtosis():.4f}  (正: 尖っている)')
print(f'95%区間  : {stats.t.interval(0.95, len(df[col])-1, df[col].mean(), stats.sem(df[col]))}')

### 正規性検定 (Shapiro-Wilk)

In [ ]:
from scipy import stats

# p > 0.05 → 正規分布に従う可能性が高い
stat, p = stats.shapiro(df['花弁長'])
print(f'統計量={stat:.4f}, p値={p:.4f}')
print('正規分布に従う' if p > 0.05 else '正規分布でない可能性あり')

### t検定 (2グループの平均差)

In [ ]:
from scipy import stats

g1 = df[df['品種'] == 0]['花弁長']
g2 = df[df['品種'] == 1]['花弁長']

# 独立2標本t検定
t, p = stats.ttest_ind(g1, g2)
print(f't={t:.4f}, p={p:.4f}')
print('有意差あり (p<0.05)' if p < 0.05 else '有意差なし')

### 一元配置分散分析 (ANOVA)

In [ ]:
from scipy import stats

# 3グループ以上の平均差を検定
groups = [group['花弁長'].values for _, group in df.groupby('品種')]
f, p = stats.f_oneway(*groups)
print(f'F={f:.4f}, p={p:.4f}')
print('グループ間に有意差あり' if p < 0.05 else 'グループ間に有意差なし')

### 相関係数

In [ ]:
from scipy import stats

# ピアソン相関係数 (線形相関)
r, p = stats.pearsonr(df['花弁長'], df['花弁幅'])
print(f'ピアソン r={r:.4f}, p={p:.4f}')

# スピアマン順位相関 (外れ値に強い)
r_sp, p_sp = stats.spearmanr(df['花弁長'], df['花弁幅'])
print(f'スピアマン r={r_sp:.4f}, p={p_sp:.4f}')

<a id='8'></a>
## 8. 機械学習

sklearn を使った分類・回帰の基本フロー。  
「前処理 → 学習 → 評価」の3ステップで進める。

### 前処理 (共通)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 特徴量とラベルを分ける
X = df[['がく片長', 'がく片幅', '花弁長', '花弁幅']]
y = df['品種']

# 学習データ / テストデータに分割 (8:2)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 標準化 (平均0・分散1)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)   # testはfit不要

print(f'学習: {X_train.shape}  テスト: {X_test.shape}')

### 分類モデル: ランダムフォレスト

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_sc, y_train)
y_pred = model.predict(X_test_sc)

print(classification_report(y_test, y_pred))

### 分類モデル: ロジスティック回帰

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

model = LogisticRegression(max_iter=500, random_state=42)
model.fit(X_train_sc, y_train)
y_pred = model.predict(X_test_sc)

print(classification_report(y_test, y_pred))

### 分類モデル: SVM

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report

model = SVC(kernel='rbf', C=1.0, random_state=42)
model.fit(X_train_sc, y_train)
y_pred = model.predict(X_test_sc)

print(classification_report(y_test, y_pred))

### 分類モデル: 勾配ブースティング

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report

model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
model.fit(X_train_sc, y_train)
y_pred = model.predict(X_test_sc)

print(classification_report(y_test, y_pred))

### 混同行列

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['品種0','品種1','品種2'],
            yticklabels=['品種0','品種1','品種2'])
plt.xlabel('予測')
plt.ylabel('実測')
plt.title('混同行列')
plt.tight_layout()
plt.show()

### クロスバリデーション

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)
scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')

print(f'各fold: {scores.round(3)}')
print(f'平均: {scores.mean():.4f} ± {scores.std():.4f}')

### 特徴量重要度

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_sc, y_train)

importances = pd.Series(model.feature_importances_, index=X.columns).sort_values()
importances.plot(kind='barh', figsize=(6, 3), color='steelblue')
plt.title('特徴量重要度')
plt.tight_layout()
plt.show()

### 回帰モデル

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_reg = df[['がく片長', 'がく片幅', '花弁幅']]
y_reg = df['花弁長']

X_tr, X_te, y_tr, y_te = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)
sc = StandardScaler()
X_tr_sc = sc.fit_transform(X_tr)
X_te_sc = sc.transform(X_te)

reg = LinearRegression()
# reg = RandomForestRegressor(n_estimators=100, random_state=42)  # ← 切り替え

reg.fit(X_tr_sc, y_tr)
y_pr = reg.predict(X_te_sc)

print(f'RMSE : {mean_squared_error(y_te, y_pr, squared=False):.4f}')
print(f'R²   : {r2_score(y_te, y_pr):.4f}')

<a id='9'></a>
## 9. 結果の保存

DataFrame・グラフ・モデルをそれぞれ保存する。

### DataFrame を CSV / Excel で保存

In [ ]:
# CSV: utf-8-sig にすると Excel で文字化けしない
df.to_csv('output.csv', index=False, encoding='utf-8-sig')
print('CSV 保存完了')

In [ ]:
# Excel (1シート)
df.to_excel('output.xlsx', index=False, sheet_name='データ')
print('Excel 保存完了')

In [ ]:
# Excel (複数シート)
summary = df.groupby('品種').mean(numeric_only=True)

with pd.ExcelWriter('output_multi.xlsx', engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='データ', index=False)
    summary.to_excel(writer, sheet_name='集計')
print('Excel (複数シート) 保存完了')

### グラフを画像で保存

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(data=df, x='品種', y='花弁長', palette='Set2', ax=ax)
ax.set_title('品種別 花弁長')
plt.tight_layout()

fig.savefig('graph.png', dpi=150, bbox_inches='tight')  # PNG
# fig.savefig('graph.pdf', bbox_inches='tight')          # PDF
print('画像保存完了')

### モデルを保存 / 読み込み

In [ ]:
import joblib

# 保存
joblib.dump(model, 'model.pkl')
print('モデル保存完了')

# 読み込み
# model = joblib.load('model.pkl')

---
## オリジナルセクション (ここから下に追加していく)

上記の一般テンプレートに加えて、自分がよく使うパターンをここに追記する。

In [ ]:
# ここに追加